Import libraries

In [10]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from dragonnet import Dragonnet
import sys
from pathlib import Path
project_root = Path("/benchmark-conversion-vs-revenue-uplift-modeling")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [11]:
%time train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 25.9 ms, sys: 4.11 ms, total: 30 ms
Wall time: 29.8 ms
CPU times: user 10.3 ms, sys: 5.05 ms, total: 15.4 ms
Wall time: 15.4 ms
CPU times: user 4.39 ms, sys: 1.97 ms, total: 6.36 ms
Wall time: 6.36 ms


In [12]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [13]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [14]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [15]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [16]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [17]:
epochs = 70
lr = 5e-5
wd = 1e-5
alpha = 1.0
beta = 1.0
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 15
shared_dropout = 0
outcome_dropout = 0 
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")

 epochs = 70
 learning rate = 5e-05
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 15
 shared hidden = 200
 outcome hidden = 100
 early stop start = 30


In [18]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    grid_alpha = trial.suggest_categorical("alpha", [0.1, 0.5, 1.0])
    grid_beta = trial.suggest_categorical("beta", [0.1, 0.5, 1.0])

    val_scores = []
    for SEED in seeds:
        seed_everything(SEED)

        dragonnet = Dragonnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            alpha=grid_alpha,
            beta=grid_beta,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_droupout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence Dragonnet epoch logs
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            dragonnet.fit(train_loader, val_loader)

        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p, _, _ = dragonnet.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)
        val_scores.append(current_val_auqc)

    mean_val_auqc = float(np.mean(val_scores))
    std_val_auqc = float(np.std(val_scores, ddof=1)) if len(val_scores) > 1 else 0.0
    trial.set_user_attr("std_Val_AUQC", std_val_auqc)
    return mean_val_auqc

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# 2. Persist search results in notebook variables
trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": t.params["lr"],
        "weight_decay": t.params["weight_decay"],
        "alpha": t.params["alpha"],
        "beta": t.params["beta"],
        "mean_Val_AUQC": float(t.value),
        "std_Val_AUQC": float(t.user_attrs.get("std_Val_AUQC", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_AUQC", ascending=False).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "alpha": float(best_params["alpha"]),
    "beta": float(best_params["beta"]),
    "mean_Val_AUQC": float(study.best_value),
    "std_Val_AUQC": float(study.best_trial.user_attrs.get("std_Val_AUQC", np.nan))
})

  0%|          | 0/50 [00:00<?, ?it/s]

Locked random seed: 412312


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 42


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 902745


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
Best trial: 23. Best value: 0.849759: 100%|██████████| 50/50 [47:13<00:00, 56.68s/it]


In [19]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 23
Best mean_Val_AUQC: 0.849759
Best params: {'lr': 0.0004813006947602241, 'weight_decay': 1.219863912496383e-05, 'alpha': 0.1, 'beta': 0.1}

Best config table:


,lr,weight_decay,alpha,beta,mean_Val_AUQC,std_Val_AUQC
0,0.000481,0.000012,0.1,0.1,0.849759,0.01948



Top 10 trials:


,trial,lr,weight_decay,alpha,beta,mean_Val_AUQC,std_Val_AUQC
0,23,0.000481,0.000012,0.1,0.1,0.849759,0.019480
1,29,0.000494,0.000024,0.1,0.1,0.849576,0.016550
2,46,0.000497,0.000097,0.1,0.1,0.849487,0.015990
3,31,0.000498,0.000016,0.1,0.1,0.848721,0.017169
4,32,0.000493,0.000016,0.1,0.1,0.848509,0.019917
5,17,0.000495,0.000015,0.1,0.1,0.848201,0.018414
6,21,0.000471,0.000013,0.1,0.1,0.847396,0.019946
7,42,0.000494,0.000019,0.1,0.1,0.847188,0.018942
8,26,0.000492,0.000018,0.1,0.1,0.847050,0.017408
9,13,0.000467,0.000010,1.0,0.1,0.846582,0.018363


In [20]:
import pandas as pd
import numpy as np
import torch

# 1. Evaluate selected config on test set (after tuning)
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

if 'best_cfg' not in globals():
    raise ValueError("best_cfg not found. Run grid-search cell first.")

best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])
best_alpha = float(best_cfg['alpha'])
best_beta = float(best_cfg['beta'])

print("Đang đánh giá trên test với best config từ validation:")
print(f"  lr={best_lr:.1e}, weight_decay={best_wd:.1e}, alpha={best_alpha:.1f}, beta={best_beta:.1f}")
print(f"Số seed: {len(seeds)}")

# 2. Loop over seeds for robust test evaluation
for SEED in seeds:
    seed_everything(SEED)

    dragonnet = Dragonnet(
        input_dim=x_men_train_t.shape[1],
        epochs=epochs,
        learning_rate=best_lr,
        alpha=best_alpha,
        beta=best_beta,
        weight_decay=best_wd,
        use_ema=ema,
        ema_alpha=ema_alpha,
        patience=patience,
        shared_hidden=shared_hidden,
        outcome_hidden=outcome_hidden,
        outcome_droupout=outcome_dropout,
        shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric,
        early_stop_start_epoch=early_stop_start,
        activation=activation
    )

    dragonnet.fit(train_loader, val_loader)

    # Test prediction
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred, _, _ = dragonnet.predict(x_men_test_t_on_device)

    uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
    y_true = y_men_test_t.detach().cpu().numpy().flatten()
    t_true = t_men_test_t.detach().cpu().numpy().flatten()

    # ATE error
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  Done Seed {SEED}")

# 3. Aggregate final test metrics
df_results = pd.DataFrame(all_runs)

print("\n" + "=" * 85)
print(f"{'CHI TIẾT TỪNG SEED (TEST SET)':^85}")
print("=" * 85)
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'KẾT QUẢ TRUNG BÌNH TEST (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
print("=" * 85)

Đang đánh giá trên test với best config từ validation:
  lr=4.8e-04, weight_decay=1.2e-05, alpha=0.1, beta=0.1
Số seed: 5
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/70 | Base Loss: 414.8980 | Tarreg Loss: 20.977810 | Total Loss: 435.8759 | Val Loss: 498.7762 | Raw Qini: 0.6780 | EMA Trend: 0.6780 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Base Loss: 537.2147 | Tarreg Loss: 27.044498 | Total Loss: 564.2592 | Val Loss: 498.3949 | Raw Qini: 0.6579 | EMA Trend: 0.6750 | (patience: 1/15)
Epoch 3/70 | Base Loss: 287.5820 | Tarreg Loss: 14.456583 | Total Loss: 302.0385 | Val Loss: 497.7483 | Raw Qini: 0.7036 | EMA Trend: 0.6792 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/70 | Base Loss: 261.0580 | Tarreg Loss: 13.230468 | Total Loss: 274.2885 | Val Loss: 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/70 | Base Loss: 422.8386 | Tarreg Loss: 21.291904 | Total Loss: 444.1305 | Val Loss: 498.3065 | Raw Qini: 0.2591 | EMA Trend: 0.2591 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Base Loss: 496.7716 | Tarreg Loss: 24.771500 | Total Loss: 521.5432 | Val Loss: 497.9116 | Raw Qini: 0.3715 | EMA Trend: 0.2759 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/70 | Base Loss: 514.5347 | Tarreg Loss: 25.711792 | Total Loss: 540.2465 | Val Loss: 497.2393 | Raw Qini: 0.4134 | EMA Trend: 0.2965 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/70 | Base Loss: 347.2964 | Tarreg Loss: 17.479147 | Total Loss: 364.7755 | Val Loss: 496.3791 | Raw Qini: 0.5878 | EMA Trend: 0.3402 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/70 | Base Loss: 356.5308 | Tarreg Loss: 17.940235 | Total Loss: 374.4711 | Val Loss: 496.0718 | Raw Qini: 0.6831 | EMA Trend: 0.3917 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/70 | Base Loss: 431.4164 | Tarreg Loss: 21.756432 | Total Loss: 453.1729 | Val Loss: 496.0442 | Raw Qini: 0.7344 | EMA Trend: 0.4431 | ⭐ NEW BEST (

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/70 | Base Loss: 421.2921 | Tarreg Loss: 22.291479 | Total Loss: 443.5836 | Val Loss: 498.5189 | Raw Qini: 0.4735 | EMA Trend: 0.4735 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Base Loss: 456.6181 | Tarreg Loss: 23.936104 | Total Loss: 480.5542 | Val Loss: 498.0499 | Raw Qini: 0.5998 | EMA Trend: 0.4925 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/70 | Base Loss: 562.2100 | Tarreg Loss: 29.185831 | Total Loss: 591.3959 | Val Loss: 497.3817 | Raw Qini: 0.6271 | EMA Trend: 0.5126 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/70 | Base Loss: 597.1566 | Tarreg Loss: 30.644949 | Total Loss: 627.8016 | Val Loss: 496.5551 | Raw Qini: 0.6318 | EMA Trend: 0.5305 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/70 | Base Loss: 403.8498 | Tarreg Loss: 21.152742 | Total Loss: 425.0025 | Val Loss: 496.1939 | Raw Qini: 0.6489 | EMA Trend: 0.5483 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/70 | Base Loss: 520.9796 | Tarreg Loss: 26.897635 | Total Loss: 547.8773 | Val Loss: 496.1886 | Raw Qini: 0.7016 | EMA Trend: 0.5713 | ⭐ NEW BEST (

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/70 | Base Loss: 408.1194 | Tarreg Loss: 20.867258 | Total Loss: 428.9867 | Val Loss: 498.3340 | Raw Qini: 0.4538 | EMA Trend: 0.4538 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Base Loss: 352.5766 | Tarreg Loss: 18.202801 | Total Loss: 370.7794 | Val Loss: 497.8989 | Raw Qini: 0.7577 | EMA Trend: 0.4993 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/70 | Base Loss: 371.7027 | Tarreg Loss: 19.143734 | Total Loss: 390.8465 | Val Loss: 497.1783 | Raw Qini: 0.7658 | EMA Trend: 0.5393 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/70 | Base Loss: 439.9175 | Tarreg Loss: 22.337486 | Total Loss: 462.2550 | Val Loss: 496.2093 | Raw Qini: 0.7544 | EMA Trend: 0.5716 | ✓ above trend but not peak (patience: 1/15)
Epoch 5/70 | Base Loss: 480.2464 | Tarreg Loss: 24.468801 | Total Loss: 504.7152 | Val Loss: 495.8036 | Raw Qini: 0.7619 | EMA Trend: 0.6001 | ✓ above trend but not peak (patience: 2/15)
Epoch 6/70 | Base Loss: 549.8821 | Tarreg Loss: 27.656910 | Total Loss: 577.5391 | Val Loss: 495.6525 | Raw Qini: 0.804

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/70 | Base Loss: 439.6072 | Tarreg Loss: 22.189453 | Total Loss: 461.7967 | Val Loss: 498.8931 | Raw Qini: 0.6979 | EMA Trend: 0.6979 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Base Loss: 335.6311 | Tarreg Loss: 17.067280 | Total Loss: 352.6984 | Val Loss: 498.4789 | Raw Qini: 0.7098 | EMA Trend: 0.6997 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/70 | Base Loss: 711.8398 | Tarreg Loss: 35.565674 | Total Loss: 747.4055 | Val Loss: 497.7816 | Raw Qini: 0.6786 | EMA Trend: 0.6965 | (patience: 1/15)
Epoch 4/70 | Base Loss: 550.8727 | Tarreg Loss: 27.771772 | Total Loss: 578.6445 | Val Loss: 496.7126 | Raw Qini: 0.7428 | EMA Trend: 0.7034 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/70 | Base Loss: 434.2457 | Tarreg Loss: 21.821753 | Total Loss: 456.0674 | Val Loss: 495.8173 | Raw Qini: 0.7914 | EMA Trend: 0.7166 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/70 | Base Loss: 372.3931 | Tarreg Loss: 18.885933 | Total Loss: 391.2790 | Val Loss: 495.6157 | Raw Qini: 0.8428 | EMA Trend: 0.7356 | ⭐ NEW BEST (peak ≥ tr

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
